In [1]:
import pandas as pd

In [2]:
from src.fred_data import FredData

# Load the three FRED datasets through the FredData class.
fred_data = FredData()
unemployment_df = fred_data.unemployment_df
cpi_df = fred_data.cpi_df
interest_df = fred_data.interest_df

# Give each dataset a name so we know which one we're looking at.
fred_datasets = [
    ("Unemployment rate", unemployment_df),
    ("Consumer Price Index", cpi_df),
    ("Federal funds rate", interest_df),
]

for name, df in fred_datasets:
    print(f"\n{name}")

    # See how big each dataset is and check what the first few rows look like.
    print("Rows and columns:", df.shape)
    display(df.head())

    # Check for missing dates or values before we put the datasets together.
    print("Missing values by column:")
    print(df.isna().sum())


Unemployment rate
Rows and columns: (946, 2)


,observation_date,UNRATE
0,1948-01-01,3.4
1,1948-02-01,3.8
2,1948-03-01,4.0
3,1948-04-01,3.9
4,1948-05-01,3.5


Missing values by column:
observation_date    0
UNRATE              1
dtype: int64

Consumer Price Index
Rows and columns: (958, 2)


,observation_date,CPIAUCSL
0,1947-01-01,21.48
1,1947-02-01,21.62
2,1947-03-01,22.00
3,1947-04-01,22.00
4,1947-05-01,21.95


Missing values by column:
observation_date    0
CPIAUCSL            1
dtype: int64

Federal funds rate
Rows and columns: (868, 2)


,observation_date,FEDFUNDS
0,1954-07-01,0.80
1,1954-08-01,1.22
2,1954-09-01,1.07
3,1954-10-01,0.85
4,1954-11-01,0.83


Missing values by column:
observation_date    0
FEDFUNDS            0
dtype: int64


In [3]:
# Check which months are missing a value before we merge the data.
for name, df in fred_datasets:
    value_column = df.columns[1]
    missing_rows = df.loc[
        df[value_column].isna(), ["observation_date", value_column]
    ]

    print(f"\n{name}")
    if missing_rows.empty:
        print("No missing values.")
    else:
        display(missing_rows)


Unemployment rate


,observation_date,UNRATE
933,2025-10-01,NaN



Consumer Price Index


,observation_date,CPIAUCSL
945,2025-10-01,NaN



Federal funds rate
No missing values.


In [4]:
# Check how far back each file goes and where it ends.
for name, df in fred_datasets:
    dates = pd.to_datetime(df["observation_date"])

    print(f"\n{name}")
    print("First month:", dates.min().strftime("%Y-%m"))
    print("Last month:", dates.max().strftime("%Y-%m"))

    # Each file should have one row per month.
    print("Duplicate dates:", dates.duplicated().sum())


Unemployment rate
First month: 1948-01
Last month: 2026-10
Duplicate dates: 0

Consumer Price Index
First month: 1947-01
Last month: 2026-10
Duplicate dates: 0

Federal funds rate
First month: 1954-07
Last month: 2026-10
Duplicate dates: 0


In [5]:
fred = fred_data.monthly_indicators()

# Check the month we found missing and what happens one year later.
display(fred.loc[["2025-10-01", "2026-10-01"]])

,UNRATE,CPIAUCSL,FEDFUNDS,inflation_yoy_pct
observation_date,,,,
2025-10-01,NaN,NaN,4.09,NaN
2026-10-01,4.1,337.476,3.88,NaN


In [6]:
from pathlib import Path

# Save the combined table in the processed data folder.
output_path = Path("data/processed/fred_monthly.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Turn the date index back into a regular column so Farah can use it.
fred.reset_index().to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Months saved:", len(fred))

Saved to: data/processed/fred_monthly.csv
Months saved: 958
